In [13]:
import numpy as np
import pandas as pd
import os, time, gc , re
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Dropout, Bidirectional, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.preprocessing import StandardScaler
from keras import backend as K
from vmdpy import VMD
import matplotlib.pyplot as plt

In [14]:
Yt = pd.read_csv('ws.csv', header=1, parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

In [15]:
# wind direction to sin/cos
Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# SD turbulence columns
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

# Turbulence intensity TI = SD / mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# Gust deviation
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Vertical shear
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Temperature and pressure
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# Time cyclic features
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

# Base feature list
features = [
    # raw wind speeds
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',
    # wind direction
    'wind_sin', 'wind_cos',
    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',
    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',
    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',
    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',
    # meteo
    'temp', 'pressure',
    # time
    'minute_sin', 'minute_cos']
target_h = "Ch1_Anem_110.00m_E_Avg_m/s"

In [16]:
np.random.seed(42)
tf.random.set_seed(42)

results = []
predictions = {}

def dataset(data_scaled, window_size=3, pred_steps=1, target_idx=0):
    X, y = [], []
    n = len(data_scaled)
    for i in range(n - window_size - pred_steps):
        X.append(data_scaled[i:i + window_size, :])
        y.append(data_scaled[i + window_size : i + window_size + pred_steps, target_idx])
    return np.array(X), np.array(y)

def mape(true, pred):
    true = np.asarray(true)
    pred = np.asarray(pred)
    return np.mean(np.abs((true - pred) / np.maximum(np.abs(true), 1e-6))) * 100

f_list = [1, 2, 3, 4, 5, 6]
w_list = [3,6,12,24]

for f in f_list:
    forecast_min = f * 10
    print(f"\n=== Forecast horizon: {f} steps ({forecast_min} min) ===")

    for target in [target_h]:
        df_y = Yt[['time'] + features].copy()
        df_y = df_y.sort_values('time').reset_index(drop=True)

        split_idx = int(0.8 * len(df_y))
        train_df = df_y[features].iloc[:split_idx]
        test_df  = df_y[features].iloc[split_idx:]

        y_train_real = train_df[[target]].values
        y_test_real  = test_df[[target]].values

        scaler_X = StandardScaler()
        train_scaled = scaler_X.fit_transform(train_df.values)
        test_scaled  = scaler_X.transform(test_df.values)

        scaler_y_real = MinMaxScaler()
        scaler_y_real.fit(y_train_real)

        target_idx = features.index(target)

        for w in w_list:
            print(f"Training target={target}: w={w} | {forecast_min}min")

            start_time = time.time()

            X_train, y_train = dataset(train_scaled, window_size=w, pred_steps=f, target_idx=target_idx)
            X_test,  y_test  = dataset(test_scaled,  window_size=w, pred_steps=f, target_idx=target_idx)

            scaler_y_model = MinMaxScaler()
            scaler_y_model.fit(y_train.reshape(-1, 1))

            y_train_s = scaler_y_model.transform(y_train.reshape(-1, 1)).reshape(-1, f)
            y_test_s  = scaler_y_model.transform(y_test.reshape(-1, 1)).reshape(-1, f)

            inp = Input(shape=(w, X_train.shape[2]))
            x = Bidirectional(
                LSTM(
                    64,
                    return_sequences=False,
                    activation='tanh',
                    recurrent_activation='sigmoid'
                )
            )(inp)
            x = Dense(64, activation='relu')(x)
            out = Dense(f, activation='linear')(x)

            model = Model(inp, out)
            model.compile(optimizer='adam', loss='mse')

            model.fit(
                X_train, y_train_s,
                epochs=150,
                batch_size=128,
                shuffle=False,
                validation_split=0.1,
                callbacks=[
                    ReduceLROnPlateau(
                        monitor='val_loss',
                        factor=0.5,
                        patience=5,
                        min_lr=1e-5,
                        verbose=1
                    ),
                    EarlyStopping(
                        monitor='val_loss',
                        patience=10,
                        restore_best_weights=True,
                        verbose=1
                    )
                ],
                verbose=1
            )

            elapsed = time.time() - start_time
            print(f"Training time: {elapsed:.2f} sec")

            y_train_pred_s = model.predict(X_train, verbose=0)
            y_test_pred_s  = model.predict(X_test,  verbose=0)

            y_train_true_real = scaler_y_real.inverse_transform(y_train_s.reshape(-1, 1)).flatten()
            y_train_pred_real = scaler_y_real.inverse_transform(y_train_pred_s.reshape(-1, 1)).flatten()
            y_test_true_real  = scaler_y_real.inverse_transform(y_test_s.reshape(-1, 1)).flatten()
            y_test_pred_real  = scaler_y_real.inverse_transform(y_test_pred_s.reshape(-1, 1)).flatten()

            rmse_train = np.sqrt(np.mean((y_train_true_real - y_train_pred_real) ** 2))
            rmse_test  = np.sqrt(np.mean((y_test_true_real  - y_test_pred_real ) ** 2))

            r2_train = r2_score(y_train_true_real, y_train_pred_real)
            r2_test  = r2_score(y_test_true_real,  y_test_pred_real)

            mae_train = mean_absolute_error(y_train_true_real, y_train_pred_real)
            mae_test  = mean_absolute_error(y_test_true_real,  y_test_pred_real)

            mape_train = mape(y_train_true_real, y_train_pred_real)
            mape_test  = mape(y_test_true_real,  y_test_pred_real)

            print(f"Train RMSE={rmse_train:.6f}, Test RMSE={rmse_test:.6f}")
            print(f"Train R2  ={r2_train:.6f}, Test R2  ={r2_test:.6f}")
            print(f"Train MAE ={mae_train:.6f}, Test MAE ={mae_test:.6f}")
            print(f"Train MAPE={mape_train:.4f}%, Test MAPE={mape_test:.4f}%")

            results.append({
                "Height": target,
                "Window": w,
                "Forecast_Min": forecast_min,
                "Train_RMSE": float(rmse_train),
                "Test_RMSE": float(rmse_test),
                "Train_R2": float(r2_train),
                "Test_R2": float(r2_test),
                "Train_MAE": float(mae_train),
                "Test_MAE": float(mae_test),
                "Train_MAPE": float(mape_train),
                "Test_MAPE": float(mape_test),
                "Time_sec": float(elapsed)
            })

            predictions[(w, f, target)] = {
                "train_true": y_train_true_real.copy(),
                "train_pred": y_train_pred_real.copy(),
                "test_true":  y_test_true_real.copy(),
                "test_pred":  y_test_pred_real.copy()
            }

            K.clear_session()
            gc.collect()


=== Forecast horizon: 1 steps (10 min) ===
Training target=Ch1_Anem_110.00m_E_Avg_m/s: w=3 | 10min
Epoch 1/150
306/306 [==============================] - 9s 19ms/step - loss: 0.0072 - val_loss: 0.0045 - lr: 0.0010
Epoch 2/150
306/306 [==============================] - 4s 12ms/step - loss: 0.0023 - val_loss: 0.0027 - lr: 0.0010
Epoch 3/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0022 - val_loss: 0.0026 - lr: 0.0010
Epoch 4/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0022 - val_loss: 0.0028 - lr: 0.0010
Epoch 5/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0021 - val_loss: 0.0029 - lr: 0.0010
Epoch 6/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0025 - val_loss: 0.0019 - lr: 0.0010
Epoch 7/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0017 - val_loss: 0.0024 - lr: 0.0010
Epoch 8/150
306/306 [==============================] - 3s 11ms/step - loss: 0.0019 - val_loss: 

In [17]:
def estimate_lag(true, pred, max_lag=20):
    lags = range(-max_lag, max_lag + 1)
    corrs = []
    true = true - np.mean(true)
    pred = pred - np.mean(pred)
    for lag in lags:
        if lag < 0:
            corr = np.corrcoef(true[:lag], pred[-lag:])[0, 1]
        elif lag > 0:
            corr = np.corrcoef(true[lag:], pred[:-lag])[0, 1]
        else:
            corr = np.corrcoef(true, pred)[0, 1]
        corrs.append(corr)
    best_lag = lags[np.nanargmax(corrs)]
    return best_lag
lag_results = []

for f, data in predictions.items():
    true_vals = data["test_true"]
    pred_vals = data["test_pred"]

    lag = estimate_lag(true_vals, pred_vals, max_lag=30)

    lag_results.append({
        "Forecast_Min": f * 10,
        "Lag_steps": lag
    })
df_lag = pd.DataFrame(lag_results)
display(df_lag)

,Forecast_Min,Lag_steps
0,"(3, 1, Ch1_Anem_110.00m_E_Avg_m/s, 3, 1, Ch1_A...",-1
1,"(6, 1, Ch1_Anem_110.00m_E_Avg_m/s, 6, 1, Ch1_A...",-1
2,"(12, 1, Ch1_Anem_110.00m_E_Avg_m/s, 12, 1, Ch1...",-1
3,"(24, 1, Ch1_Anem_110.00m_E_Avg_m/s, 24, 1, Ch1...",-1
4,"(3, 2, Ch1_Anem_110.00m_E_Avg_m/s, 3, 2, Ch1_A...",-3
5,"(6, 2, Ch1_Anem_110.00m_E_Avg_m/s, 6, 2, Ch1_A...",-3
6,"(12, 2, Ch1_Anem_110.00m_E_Avg_m/s, 12, 2, Ch1...",-3
7,"(24, 2, Ch1_Anem_110.00m_E_Avg_m/s, 24, 2, Ch1...",-3
8,"(3, 3, Ch1_Anem_110.00m_E_Avg_m/s, 3, 3, Ch1_A...",-5
9,"(6, 3, Ch1_Anem_110.00m_E_Avg_m/s, 6, 3, Ch1_A...",-5


In [18]:
base_dir = "2-50m"
os.makedirs(base_dir, exist_ok=True)
safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_h)
for (w, f, target), data in predictions.items():
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)
    df_test = pd.DataFrame({
        "True_Test": data["test_true"],
        "Pred_Test": data["test_pred"]})
    filename = f"{safe_h}_test_{f*10}min.csv"
    df_test.to_csv(os.path.join(w_dir, filename), index=False)

df_results = pd.DataFrame(results)
for w in df_results["Window"].unique():
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)
    df_w = df_results[df_results["Window"] == w]
    df_w.to_csv(os.path.join(w_dir, f"summary_w{w}.csv"), index=False)


colors = ["#006699", "#b30000", "#009933",
          "#ff9900", "#660066", "#666600"]

plt.rcParams["font.size"] = 13

def plot_saved_by_w(predictions_dict, base_dir):

    for idx, ((w, f, target), data) in enumerate(predictions_dict.items()):
        w_dir = os.path.join(base_dir, f"w{w}")
        os.makedirs(w_dir, exist_ok=True)

        true_vals = data["test_true"]
        pred_vals = data["test_pred"]

        rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
        r2 = r2_score(true_vals, pred_vals)

        forecast_min = f * 10

        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())

        plt.figure(figsize=(7, 7))
        plt.scatter(
            true_vals,
            pred_vals,
            alpha=0.35,
            color=colors[idx % len(colors)],
            edgecolor="none"
        )

        plt.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Ideal Fit"
        )

        plt.xlabel("True Wind Speed (m/s)")
        plt.ylabel("Predicted Wind Speed (m/s)")
        plt.title(f"{forecast_min}-Minute Ahead Prediction (w={w})")

        plt.text(
            min_val,
            max_val,
            f"$R^2 = {r2:.4f}$\nRMSE = {rmse:.4f}",
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.85)
        )

        plt.grid(alpha=0.35)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            os.path.join(w_dir, f"scatter_{forecast_min}min.png"),
            dpi=300
        )
        plt.close()

plot_saved_by_w(predictions, base_dir="2-50m")